In [ ]:
!pip install box2d-py
!brew install ffmpeg
!brew install freeglut
!brew install pyglet
!brew install pyopengl
!pip install pyvirtualdisplay
!pip -q install pyglet
!pip -q install pyopengl
!pip install swig
!pip install pybullet

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import numpy as np
import os
import glob
from pyvirtualdisplay import Display
from gymnasium.wrappers.monitoring.video_recorder import VideoRecorder

    Bipedal Walker Class

In [ ]:
import math
from typing import TYPE_CHECKING, List, Optional

import numpy as np

import gym
from gym import error, spaces
from gym.error import DependencyNotInstalled
from gym.utils import EzPickle

try:
    import Box2D
    from Box2D.b2 import (
        circleShape,
        contactListener,
        edgeShape,
        fixtureDef,
        polygonShape,
        revoluteJointDef,
    )
except ImportError:
    raise DependencyNotInstalled("box2D is not installed, run `pip install gym[box2d]`")


if TYPE_CHECKING:
    import pygame

FPS = 50
SCALE = 30.0  # affects how fast-paced the game is, forces should be adjusted as well

MOTORS_TORQUE = 80
SPEED_HIP = 4
SPEED_KNEE = 6
LIDAR_RANGE = 160 / SCALE

INITIAL_RANDOM = 5

HULL_POLY = [(-30, +9), (+6, +9), (+34, +1), (+34, -8), (-30, -8)]
LEG_DOWN = -8 / SCALE
LEG_W, LEG_H = 8 / SCALE, 34 / SCALE

VIEWPORT_W = 600
VIEWPORT_H = 400

TERRAIN_STEP = 14 / SCALE
TERRAIN_LENGTH = 200  # in steps
TERRAIN_HEIGHT = VIEWPORT_H / SCALE / 4
TERRAIN_GRASS = 10  # low long are grass spots, in steps
TERRAIN_STARTPAD = 20  # in steps
FRICTION = 2.5

HULL_FD = fixtureDef(
    shape=polygonShape(vertices=[(x / SCALE, y / SCALE) for x, y in HULL_POLY]),
    density=5.0,
    friction=0.1,
    categoryBits=0x0020,
    maskBits=0x001,  # collide only with ground
    restitution=0.0,
)  # 0.99 bouncy

LEG_FD = fixtureDef(
    shape=polygonShape(box=(LEG_W / 2, LEG_H / 2)),
    density=1.0,
    restitution=0.0,
    categoryBits=0x0020,
    maskBits=0x001,
)

LOWER_FD = fixtureDef(
    shape=polygonShape(box=(0.8 * LEG_W / 2, LEG_H / 2)),
    density=1.0,
    restitution=0.0,
    categoryBits=0x0020,
    maskBits=0x001,
)


class ContactDetector(contactListener):
    def __init__(self, env):
        contactListener.__init__(self)
        self.env = env

    def BeginContact(self, contact):
        if (
            self.env.hull == contact.fixtureA.body
            or self.env.hull == contact.fixtureB.body
        ):
            self.env.game_over = True
        for leg in [self.env.legs[1], self.env.legs[3]]:
            if leg in [contact.fixtureA.body, contact.fixtureB.body]:
                leg.ground_contact = True

    def EndContact(self, contact):
        for leg in [self.env.legs[1], self.env.legs[3]]:
            if leg in [contact.fixtureA.body, contact.fixtureB.body]:
                leg.ground_contact = False

In [ ]:
class BipedalWalker4(gym.Env, EzPickle):
    """
    ### Description
    This is a simple 4-joint walker robot environment.
    There are two versions:
    - Normal, with slightly uneven terrain.
    - Hardcore, with ladders, stumps, pitfalls.

    To solve the normal version, you need to get 300 points in 1600 time steps.
    To solve the hardcore version, you need 300 points in 2000 time steps.

    A heuristic is provided for testing. It's also useful to get demonstrations
    to learn from. To run the heuristic:
    ```
    python gym/envs/box2d/bipedal_walker.py
    ```

    ### Action Space
    Actions are motor speed values in the [-1, 1] range for each of the
    4 joints at both hips and knees.

    ### Observation Space
    State consists of hull angle speed, angular velocity, horizontal speed,
    vertical speed, position of joints and joints angular speed, legs contact
    with ground, and 10 lidar rangefinder measurements. There are no coordinates
    in the state vector.

    ### Rewards
    Reward is given for moving forward, totaling 300+ points up to the far end.
    If the robot falls, it gets -100. Applying motor torque costs a small
    amount of points. A more optimal agent will get a better score.

    ### Starting State
    The walker starts standing at the left end of the terrain with the hull
    horizontal, and both legs in the same position with a slight knee angle.

    ### Episode Termination
    The episode will terminate if the hull gets in contact with the ground or
    if the walker exceeds the right end of the terrain length.

    ### Arguments
    To use to the _hardcore_ environment, you need to specify the
    `hardcore=True` argument like below:
    ```python
    import gym
    env = gym.make("BipedalWalker-v3", hardcore=True)
    ```

    ### Version History
    - v3: returns closest lidar trace instead of furthest;
        faster video recording
    - v2: Count energy spent
    - v1: Legs now report contact with ground; motors have higher torque and
        speed; ground has higher friction; lidar rendered less nervously.
    - v0: Initial version


    <!-- ### References -->

    ### Credits
    Created by Oleg Klimov

    """

    metadata = {
        "render_modes": ["human", "rgb_array"],
        "render_fps": FPS,
    }

    def __init__(self, render_mode: Optional[str] = None, hardcore: bool = False):
        EzPickle.__init__(self, render_mode, hardcore)
        self.isopen = True

        self.world = Box2D.b2World()
        self.terrain: List[Box2D.b2Body] = []
        self.hull: Optional[Box2D.b2Body] = None

        self.prev_shaping = None

        self.hardcore = hardcore

        self.fd_polygon = fixtureDef(
            shape=polygonShape(vertices=[(0, 0), (1, 0), (1, -1), (0, -1)]),
            friction=FRICTION,
        )

        self.fd_edge = fixtureDef(
            shape=edgeShape(vertices=[(0, 0), (1, 1)]),
            friction=FRICTION,
            categoryBits=0x0001,
        )

        # we use 5.0 to represent the joints moving at maximum
        # 5 x the rated speed due to impulses from ground contact etc.
        low = np.array(
            [
                -math.pi,
                -5.0,
                -5.0,
                -5.0,
                -math.pi,
                -5.0,
                -math.pi,
                -5.0,
                -0.0,
                -math.pi,
                -5.0,
                -math.pi,
                -5.0,
                -0.0,
                -5000.0,
                -10000.0,
                -20000.0,
                -30000.0
            ]
            + [-1.0] * 10
        ).astype(np.float32)
        high = np.array(
            [
                math.pi,
                5.0,
                5.0,
                5.0,
                math.pi,
                5.0,
                math.pi,
                5.0,
                5.0,
                math.pi,
                5.0,
                math.pi,
                5.0,
                5.0,
                5000.0,
                10000.0,
                20000.0,
                30000.0
            ]
            + [1.0] * 10
        ).astype(np.float32)
        self.action_space = spaces.Box(
            np.array([-1, -1, -1, -1]).astype(np.float32),
            np.array([1, 1, 1, 1]).astype(np.float32),
        )
        self.observation_space = spaces.Box(low, high)

        # state = [
        #     self.hull.angle,  # Normal angles up to 0.5 here, but sure more is possible.
        #     2.0 * self.hull.angularVelocity / FPS,
        #     0.3 * vel.x * (VIEWPORT_W / SCALE) / FPS,  # Normalized to get -1..1 range
        #     0.3 * vel.y * (VIEWPORT_H / SCALE) / FPS,
        #     self.joints[
        #         0
        #     ].angle,  # This will give 1.1 on high up, but it's still OK (and there should be spikes on hiting the ground, that's normal too)
        #     self.joints[0].speed / SPEED_HIP,
        #     self.joints[1].angle + 1.0,
        #     self.joints[1].speed / SPEED_KNEE,
        #     1.0 if self.legs[1].ground_contact else 0.0,
        #     self.joints[2].angle,
        #     self.joints[2].speed / SPEED_HIP,
        #     self.joints[3].angle + 1.0,
        #     self.joints[3].speed / SPEED_KNEE,
        #     1.0 if self.legs[3].ground_contact else 0.0,
        # ]
        # state += [l.fraction for l in self.lidar]

        self.render_mode = render_mode
        self.screen: Optional[pygame.Surface] = None
        self.clock = None

    def _destroy(self):
        if not self.terrain:
            return
        self.world.contactListener = None
        for t in self.terrain:
            self.world.DestroyBody(t)
        self.terrain = []
        self.world.DestroyBody(self.hull)
        self.hull = None
        for leg in self.legs:
            self.world.DestroyBody(leg)
        self.legs = []
        self.joints = []

    def _generate_terrain(self, hardcore):
        GRASS, STUMP, STAIRS, PIT, _STATES_ = range(5)
        state = GRASS
        velocity = 0.0
        y = TERRAIN_HEIGHT
        counter = TERRAIN_STARTPAD
        oneshot = False
        self.terrain = []
        self.terrain_x = []
        self.terrain_y = []

        stair_steps, stair_width, stair_height = 0, 0, 0
        original_y = 0
        for i in range(TERRAIN_LENGTH):
            x = i * TERRAIN_STEP
            self.terrain_x.append(x)

            if state == GRASS and not oneshot:
                velocity = 0.8 * velocity + 0.01 * np.sign(TERRAIN_HEIGHT - y)
                if i > TERRAIN_STARTPAD:
                    velocity += self.np_random.uniform(-1, 1) / SCALE  # 1
                y += velocity

            elif state == PIT and oneshot:
                counter = self.np_random.integers(3, 5)
                poly = [
                    (x, y),
                    (x + TERRAIN_STEP, y),
                    (x + TERRAIN_STEP, y - 4 * TERRAIN_STEP),
                    (x, y - 4 * TERRAIN_STEP),
                ]
                self.fd_polygon.shape.vertices = poly
                t = self.world.CreateStaticBody(fixtures=self.fd_polygon)
                t.color1, t.color2 = (255, 255, 255), (153, 153, 153)
                self.terrain.append(t)

                self.fd_polygon.shape.vertices = [
                    (p[0] + TERRAIN_STEP * counter, p[1]) for p in poly
                ]
                t = self.world.CreateStaticBody(fixtures=self.fd_polygon)
                t.color1, t.color2 = (255, 255, 255), (153, 153, 153)
                self.terrain.append(t)
                counter += 2
                original_y = y

            elif state == PIT and not oneshot:
                y = original_y
                if counter > 1:
                    y -= 4 * TERRAIN_STEP

            elif state == STUMP and oneshot:
                counter = self.np_random.integers(1, 3)
                poly = [
                    (x, y),
                    (x + counter * TERRAIN_STEP, y),
                    (x + counter * TERRAIN_STEP, y + counter * TERRAIN_STEP),
                    (x, y + counter * TERRAIN_STEP),
                ]
                self.fd_polygon.shape.vertices = poly
                t = self.world.CreateStaticBody(fixtures=self.fd_polygon)
                t.color1, t.color2 = (255, 255, 255), (153, 153, 153)
                self.terrain.append(t)

            elif state == STAIRS and oneshot:
                stair_height = +1 if self.np_random.random() > 0.5 else -1
                stair_width = self.np_random.integers(4, 5)
                stair_steps = self.np_random.integers(3, 5)
                original_y = y
                for s in range(stair_steps):
                    poly = [
                        (
                            x + (s * stair_width) * TERRAIN_STEP,
                            y + (s * stair_height) * TERRAIN_STEP,
                        ),
                        (
                            x + ((1 + s) * stair_width) * TERRAIN_STEP,
                            y + (s * stair_height) * TERRAIN_STEP,
                        ),
                        (
                            x + ((1 + s) * stair_width) * TERRAIN_STEP,
                            y + (-1 + s * stair_height) * TERRAIN_STEP,
                        ),
                        (
                            x + (s * stair_width) * TERRAIN_STEP,
                            y + (-1 + s * stair_height) * TERRAIN_STEP,
                        ),
                    ]
                    self.fd_polygon.shape.vertices = poly
                    t = self.world.CreateStaticBody(fixtures=self.fd_polygon)
                    t.color1, t.color2 = (255, 255, 255), (153, 153, 153)
                    self.terrain.append(t)
                counter = stair_steps * stair_width

            elif state == STAIRS and not oneshot:
                s = stair_steps * stair_width - counter - stair_height
                n = s / stair_width
                y = original_y + (n * stair_height) * TERRAIN_STEP

            oneshot = False
            self.terrain_y.append(y)
            counter -= 1
            if counter == 0:
                counter = self.np_random.integers(TERRAIN_GRASS / 2, TERRAIN_GRASS)
                if state == GRASS and hardcore:
                    state = self.np_random.integers(1, _STATES_)
                    oneshot = True
                else:
                    state = GRASS
                    oneshot = True

        self.terrain_poly = []
        for i in range(TERRAIN_LENGTH - 1):
            poly = [
                (self.terrain_x[i], self.terrain_y[i]),
                (self.terrain_x[i + 1], self.terrain_y[i + 1]),
            ]
            self.fd_edge.shape.vertices = poly
            t = self.world.CreateStaticBody(fixtures=self.fd_edge)
            color = (76, 255 if i % 2 == 0 else 204, 76)
            t.color1 = color
            t.color2 = color
            self.terrain.append(t)
            color = (102, 153, 76)
            poly += [(poly[1][0], 0), (poly[0][0], 0)]
            self.terrain_poly.append((poly, color))
        self.terrain.reverse()

    def _generate_clouds(self):
        # Sorry for the clouds, couldn't resist
        self.cloud_poly = []
        for i in range(TERRAIN_LENGTH // 20):
            x = self.np_random.uniform(0, TERRAIN_LENGTH) * TERRAIN_STEP
            y = VIEWPORT_H / SCALE * 3 / 4
            poly = [
                (
                    x
                    + 15 * TERRAIN_STEP * math.sin(3.14 * 2 * a / 5)
                    + self.np_random.uniform(0, 5 * TERRAIN_STEP),
                    y
                    + 5 * TERRAIN_STEP * math.cos(3.14 * 2 * a / 5)
                    + self.np_random.uniform(0, 5 * TERRAIN_STEP),
                )
                for a in range(5)
            ]
            x1 = min(p[0] for p in poly)
            x2 = max(p[0] for p in poly)
            self.cloud_poly.append((poly, x1, x2))

    def reset(
        self,
        *,
        seed: Optional[int] = None,
        options: Optional[dict] = None,
    ):
        super().reset(seed=seed)
        self._destroy()
        self.world.contactListener_bug_workaround = ContactDetector(self)
        self.world.contactListener = self.world.contactListener_bug_workaround
        self.game_over = False
        self.prev_shaping = None
        self.scroll = 0.0
        self.lidar_render = 0

        self._generate_terrain(self.hardcore)
        self._generate_clouds()

        init_x = TERRAIN_STEP * TERRAIN_STARTPAD / 2
        init_y = TERRAIN_HEIGHT + 2 * LEG_H
        self.hull = self.world.CreateDynamicBody(
            position=(init_x, init_y), fixtures=HULL_FD
        )
        self.hull.color1 = (127, 51, 229)
        self.hull.color2 = (76, 76, 127)
        self.hull.ApplyForceToCenter(
            (self.np_random.uniform(-INITIAL_RANDOM, INITIAL_RANDOM), 0), True
        )

        self.legs: List[Box2D.b2Body] = []
        self.joints: List[Box2D.b2RevoluteJoint] = []
        for i in [-1, +1]:
            leg = self.world.CreateDynamicBody(
                position=(init_x, init_y - LEG_H / 2 - LEG_DOWN),
                angle=(i * 0.05),
                fixtures=LEG_FD,
            )
            leg.color1 = (153 - i * 25, 76 - i * 25, 127 - i * 25)
            leg.color2 = (102 - i * 25, 51 - i * 25, 76 - i * 25)
            rjd = revoluteJointDef(
                bodyA=self.hull,
                bodyB=leg,
                localAnchorA=(0, LEG_DOWN),
                localAnchorB=(0, LEG_H / 2),
                enableMotor=True,
                enableLimit=True,
                maxMotorTorque=MOTORS_TORQUE,
                motorSpeed=i,
                lowerAngle=-0.8,
                upperAngle=1.1,
            )
            self.legs.append(leg)
            self.joints.append(self.world.CreateJoint(rjd))

            lower = self.world.CreateDynamicBody(
                position=(init_x, init_y - LEG_H * 3 / 2 - LEG_DOWN),
                angle=(i * 0.05),
                fixtures=LOWER_FD,
            )
            lower.color1 = (153 - i * 25, 76 - i * 25, 127 - i * 25)
            lower.color2 = (102 - i * 25, 51 - i * 25, 76 - i * 25)
            rjd = revoluteJointDef(
                bodyA=leg,
                bodyB=lower,
                localAnchorA=(0, -LEG_H / 2),
                localAnchorB=(0, LEG_H / 2),
                enableMotor=True,
                enableLimit=True,
                maxMotorTorque=MOTORS_TORQUE,
                motorSpeed=1,
                lowerAngle=-1.6,
                upperAngle=-0.1,
            )
            lower.ground_contact = False
            self.legs.append(lower)
            self.joints.append(self.world.CreateJoint(rjd))

        self.drawlist = self.terrain + self.legs + [self.hull]

        class LidarCallback(Box2D.b2.rayCastCallback):
            def ReportFixture(self, fixture, point, normal, fraction):
                if (fixture.filterData.categoryBits & 1) == 0:
                    return -1
                self.p2 = point
                self.fraction = fraction
                return fraction

        self.lidar = [LidarCallback() for _ in range(10)]
        if self.render_mode == "human":
            self.render()
        return self.step(np.array([0, 0, 0, 0]))[0], {}

    def step(self, action: np.ndarray):
        assert self.hull is not None

        # self.hull.ApplyForceToCenter((0, 20), True) -- Uncomment this to receive a bit of stability help
        control_speed = False  # Should be easier as well
        if control_speed:
            self.joints[0].motorSpeed = float(SPEED_HIP * np.clip(action[0], -1, 1))
            self.joints[1].motorSpeed = float(SPEED_KNEE * np.clip(action[1], -1, 1))
            self.joints[2].motorSpeed = float(SPEED_HIP * np.clip(action[2], -1, 1))
            self.joints[3].motorSpeed = float(SPEED_KNEE * np.clip(action[3], -1, 1))
        else:
            self.joints[0].motorSpeed = float(SPEED_HIP * np.sign(action[0]))
            self.joints[0].maxMotorTorque = float(
                MOTORS_TORQUE * np.clip(np.abs(action[0]), 0, 1)
            )
            self.joints[1].motorSpeed = float(SPEED_KNEE * np.sign(action[1]))
            self.joints[1].maxMotorTorque = float(
                MOTORS_TORQUE * np.clip(np.abs(action[1]), 0, 1)
            )
            self.joints[2].motorSpeed = float(SPEED_HIP * np.sign(action[2]))
            self.joints[2].maxMotorTorque = float(
                MOTORS_TORQUE * np.clip(np.abs(action[2]), 0, 1)
            )
            self.joints[3].motorSpeed = float(SPEED_KNEE * np.sign(action[3]))
            self.joints[3].maxMotorTorque = float(
                MOTORS_TORQUE * np.clip(np.abs(action[3]), 0, 1)
            )

        self.world.Step(1.0 / FPS, 6 * 30, 2 * 30)

        pos = self.hull.position
        vel = self.hull.linearVelocity
        #Add imposter 
        # Generate impostor features
        impostor_1 = np.random.uniform(-5000.0, 5000.0)
        impostor_2 = np.random.uniform(-10000.0, 10000.0)
        impostor_3 = np.random.uniform(-20000.0, 20000.0)
        impostor_4 = np.random.uniform(-30000.0, 30000.0)
          

        for i in range(10):
            self.lidar[i].fraction = 1.0
            self.lidar[i].p1 = pos
            self.lidar[i].p2 = (
                pos[0] + math.sin(1.5 * i / 10.0) * LIDAR_RANGE,
                pos[1] - math.cos(1.5 * i / 10.0) * LIDAR_RANGE,
            )
            self.world.RayCast(self.lidar[i], self.lidar[i].p1, self.lidar[i].p2)
        

        state = [
            self.hull.angle,  # Normal angles up to 0.5 here, but sure more is possible.
            2.0 * self.hull.angularVelocity / FPS,
            0.3 * vel.x * (VIEWPORT_W / SCALE) / FPS,  # Normalized to get -1..1 range
            0.3 * vel.y * (VIEWPORT_H / SCALE) / FPS,
            self.joints[0].angle,
            # This will give 1.1 on high up, but it's still OK (and there should be spikes on hiting the ground, that's normal too)
            self.joints[0].speed / SPEED_HIP,
            self.joints[1].angle + 1.0,
            self.joints[1].speed / SPEED_KNEE,
            1.0 if self.legs[1].ground_contact else 0.0,
            self.joints[2].angle,
            self.joints[2].speed / SPEED_HIP,
            self.joints[3].angle + 1.0,
            self.joints[3].speed / SPEED_KNEE,
            1.0 if self.legs[3].ground_contact else 0.0,
            impostor_1, #14
            impostor_2, #15
            impostor_3, #16
            impostor_4, #17
        ]
        state += [l.fraction for l in self.lidar]
        assert len(state) == 28

        self.scroll = pos.x - VIEWPORT_W / SCALE / 5

        shaping = (
            130 * pos[0] / SCALE
        )  # moving forward is a way to receive reward (normalized to get 300 on completion)
        shaping -= 5.0 * abs(
            state[0]
            - 100 * state[14]
            - 100 * state[15]
            - 100 * state[16]
            - 100 * state[17]
        )  # keep head straight, other than that and falling, any behavior is unpunished

        reward = 0
        if self.prev_shaping is not None:
            reward = shaping - self.prev_shaping
        self.prev_shaping = shaping

        for a in action:
            reward -= 0.00035 * MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
            # normalized to about -50.0 using heuristic, more optimal agent should spend less

        terminated = False
        if self.game_over or pos[0] < 0:
            reward = -100
            terminated = True
        if pos[0] > (TERRAIN_LENGTH - TERRAIN_GRASS) * TERRAIN_STEP:
            terminated = True

        if self.render_mode == "human":
            self.render()
        return np.array(state, dtype=np.float32), reward, terminated, False, {}

    def render(self):
        if self.render_mode is None:
            gym.logger.warn(
                "You are calling render method without specifying any render mode. "
                "You can specify the render_mode at initialization, "
                f'e.g. gym("{self.spec.id}", render_mode="rgb_array")'
            )
            return

        try:
            import pygame
            from pygame import gfxdraw
        except ImportError:
            raise DependencyNotInstalled(
                "pygame is not installed, run `pip install gym[box2d]`"
            )

        if self.screen is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((VIEWPORT_W, VIEWPORT_H))
        if self.clock is None:
            self.clock = pygame.time.Clock()

        self.surf = pygame.Surface(
            (VIEWPORT_W + max(0.0, self.scroll) * SCALE, VIEWPORT_H)
        )

        pygame.transform.scale(self.surf, (SCALE, SCALE))

        pygame.draw.polygon(
            self.surf,
            color=(215, 215, 255),
            points=[
                (self.scroll * SCALE, 0),
                (self.scroll * SCALE + VIEWPORT_W, 0),
                (self.scroll * SCALE + VIEWPORT_W, VIEWPORT_H),
                (self.scroll * SCALE, VIEWPORT_H),
            ],
        )

        for poly, x1, x2 in self.cloud_poly:
            if x2 < self.scroll / 2:
                continue
            if x1 > self.scroll / 2 + VIEWPORT_W / SCALE:
                continue
            pygame.draw.polygon(
                self.surf,
                color=(255, 255, 255),
                points=[
                    (p[0] * SCALE + self.scroll * SCALE / 2, p[1] * SCALE) for p in poly
                ],
            )
            gfxdraw.aapolygon(
                self.surf,
                [(p[0] * SCALE + self.scroll * SCALE / 2, p[1] * SCALE) for p in poly],
                (255, 255, 255),
            )
        for poly, color in self.terrain_poly:
            if poly[1][0] < self.scroll:
                continue
            if poly[0][0] > self.scroll + VIEWPORT_W / SCALE:
                continue
            scaled_poly = []
            for coord in poly:
                scaled_poly.append([coord[0] * SCALE, coord[1] * SCALE])
            pygame.draw.polygon(self.surf, color=color, points=scaled_poly)
            gfxdraw.aapolygon(self.surf, scaled_poly, color)

        self.lidar_render = (self.lidar_render + 1) % 100
        i = self.lidar_render
        if i < 2 * len(self.lidar):
            single_lidar = (
                self.lidar[i]
                if i < len(self.lidar)
                else self.lidar[len(self.lidar) - i - 1]
            )
            if hasattr(single_lidar, "p1") and hasattr(single_lidar, "p2"):
                pygame.draw.line(
                    self.surf,
                    color=(255, 0, 0),
                    start_pos=(single_lidar.p1[0] * SCALE, single_lidar.p1[1] * SCALE),
                    end_pos=(single_lidar.p2[0] * SCALE, single_lidar.p2[1] * SCALE),
                    width=1,
                )

        for obj in self.drawlist:
            for f in obj.fixtures:
                trans = f.body.transform
                if type(f.shape) is circleShape:
                    pygame.draw.circle(
                        self.surf,
                        color=obj.color1,
                        center=trans * f.shape.pos * SCALE,
                        radius=f.shape.radius * SCALE,
                    )
                    pygame.draw.circle(
                        self.surf,
                        color=obj.color2,
                        center=trans * f.shape.pos * SCALE,
                        radius=f.shape.radius * SCALE,
                    )
                else:
                    path = [trans * v * SCALE for v in f.shape.vertices]
                    if len(path) > 2:
                        pygame.draw.polygon(self.surf, color=obj.color1, points=path)
                        gfxdraw.aapolygon(self.surf, path, obj.color1)
                        path.append(path[0])
                        pygame.draw.polygon(
                            self.surf, color=obj.color2, points=path, width=1
                        )
                        gfxdraw.aapolygon(self.surf, path, obj.color2)
                    else:
                        pygame.draw.aaline(
                            self.surf,
                            start_pos=path[0],
                            end_pos=path[1],
                            color=obj.color1,
                        )

        flagy1 = TERRAIN_HEIGHT * SCALE
        flagy2 = flagy1 + 50
        x = TERRAIN_STEP * 3 * SCALE
        pygame.draw.aaline(
            self.surf, color=(0, 0, 0), start_pos=(x, flagy1), end_pos=(x, flagy2)
        )
        f = [
            (x, flagy2),
            (x, flagy2 - 10),
            (x + 25, flagy2 - 5),
        ]
        pygame.draw.polygon(self.surf, color=(230, 51, 0), points=f)
        pygame.draw.lines(
            self.surf, color=(0, 0, 0), points=f + [f[0]], width=1, closed=False
        )

        self.surf = pygame.transform.flip(self.surf, False, True)

        if self.render_mode == "human":
            assert self.screen is not None
            self.screen.blit(self.surf, (-self.scroll * SCALE, 0))
            pygame.event.pump()
            self.clock.tick(self.metadata["render_fps"])
            pygame.display.flip()
        elif self.render_mode == "rgb_array":
            return np.transpose(
                np.array(pygame.surfarray.pixels3d(self.surf)), axes=(1, 0, 2)
            )[:, -VIEWPORT_W:]

    def close(self):
        if self.screen is not None:
            import pygame

            pygame.display.quit()
            pygame.quit()
            self.isopen = False


In [ ]:
if __name__ == "__main__":
    # Heurisic: suboptimal, have no notion of balance.
    env = BipedalWalker4()
    env.reset()
    steps = 0
    total_reward = 0
    a = np.array([0.0, 0.0, 0.0, 0.0])
    STAY_ON_ONE_LEG, PUT_OTHER_DOWN, PUSH_OFF = 1, 2, 3
    SPEED = 0.29  # Will fall forward on higher speed
    state = STAY_ON_ONE_LEG
    moving_leg = 0
    supporting_leg = 1 - moving_leg
    SUPPORT_KNEE_ANGLE = +0.1
    supporting_knee_angle = SUPPORT_KNEE_ANGLE
    while True:
        s, r, terminated, truncated, info = env.step(a)
        total_reward += r
        if steps % 20 == 0 or terminated or truncated:
            print("\naction " + str([f"{x:+0.2f}" for x in a]))
            print(f"step {steps} total_reward {total_reward:+0.2f}")
            print("hull " + str([f"{x:+0.2f}" for x in s[0:4]]))
            print("leg0 " + str([f"{x:+0.2f}" for x in s[4:9]]))
            print("leg1 " + str([f"{x:+0.2f}" for x in s[9:14]]))
        steps += 1

        contact0 = s[8]
        contact1 = s[13]
        moving_s_base = 4 + 5 * moving_leg
        supporting_s_base = 4 + 5 * supporting_leg

        hip_targ = [None, None]  # -0.8 .. +1.1
        knee_targ = [None, None]  # -0.6 .. +0.9
        hip_todo = [0.0, 0.0]
        knee_todo = [0.0, 0.0]

        if state == STAY_ON_ONE_LEG:
            hip_targ[moving_leg] = 1.1
            knee_targ[moving_leg] = -0.6
            supporting_knee_angle += 0.03
            if s[2] > SPEED:
                supporting_knee_angle += 0.03
            supporting_knee_angle = min(supporting_knee_angle, SUPPORT_KNEE_ANGLE)
            knee_targ[supporting_leg] = supporting_knee_angle
            if s[supporting_s_base + 0] < 0.10:  # supporting leg is behind
                state = PUT_OTHER_DOWN
        if state == PUT_OTHER_DOWN:
            hip_targ[moving_leg] = +0.1
            knee_targ[moving_leg] = SUPPORT_KNEE_ANGLE
            knee_targ[supporting_leg] = supporting_knee_angle
            if s[moving_s_base + 4]:
                state = PUSH_OFF
                supporting_knee_angle = min(s[moving_s_base + 2], SUPPORT_KNEE_ANGLE)
        if state == PUSH_OFF:
            knee_targ[moving_leg] = supporting_knee_angle
            knee_targ[supporting_leg] = +1.0
            if s[supporting_s_base + 2] > 0.88 or s[2] > 1.2 * SPEED:
                state = STAY_ON_ONE_LEG
                moving_leg = 1 - moving_leg
                supporting_leg = 1 - moving_leg

        if hip_targ[0]:
            hip_todo[0] = 0.9 * (hip_targ[0] - s[4]) - 0.25 * s[5]
        if hip_targ[1]:
            hip_todo[1] = 0.9 * (hip_targ[1] - s[9]) - 0.25 * s[10]
        if knee_targ[0]:
            knee_todo[0] = 4.0 * (knee_targ[0] - s[6]) - 0.25 * s[7]
        if knee_targ[1]:
            knee_todo[1] = 4.0 * (knee_targ[1] - s[11]) - 0.25 * s[12]

        hip_todo[0] -= 0.9 * (0 - s[0]) - 1.5 * s[1]  # PID to keep head strait
        hip_todo[1] -= 0.9 * (0 - s[0]) - 1.5 * s[1]
        knee_todo[0] -= 15.0 * s[3]  # vertical speed, to damp oscillations
        knee_todo[1] -= 15.0 * s[3]

        a[0] = hip_todo[0]
        a[1] = knee_todo[0]
        a[2] = hip_todo[1]
        a[3] = knee_todo[1]
        a = np.clip(0.5 * a, -1.0, 1.0)

        if terminated or truncated:
            break

    Register BipedalWalker4

In [ ]:
from gym.envs.registration import register

register(
    id='BipedalWalker4-v3',
    entry_point='__main__:BipedalWalker4',  # '__main__' because it's defined in the notebook
    max_episode_steps=1600,
)


    Callback to record observations

In [ ]:
import csv
import stable_baselines3 as sb3
from stable_baselines3.common.callbacks import BaseCallback
import pandas as pd
# Define custom callback to record observations
# class CustomTrainingCallback(BaseCallback):
#     def __init__(self, verbose=0):
#         super(CustomTrainingCallback, self).__init__(verbose)
#         self.observations = []
#         self.rewards = []
#         self.step_rewards = []


#     def _on_step(self) -> bool:
#         self.observations.append(self.locals['new_obs'])  # Access the observation
#         self.rewards.append(self.locals['rewards'])  # Access the reward
#         if 'step' not in self.locals:
#             self.locals['step'] = 0
#         self.step_rewards.append((self.locals['step'], self.locals['rewards']))
#         self.locals['step'] += 1

#         return True
# Define custom callback to record observations
# class CustomCallback(BaseCallback):
#     def __init__(self, verbose=0):
#         super(CustomCallback, self).__init__(verbose)
#         self.observations = []
#         self.step_rewards = []

#     def _on_step(self) -> bool:
#         self.observations.append(self.locals['obs'].copy())
#         self.step_rewards.append((self.num_timesteps, self.locals['rewards'][0]))
#         return True

#     def _on_training_end(self) -> None:
#         pass    

class CustomCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(CustomCallback, self).__init__(verbose)
        self.observations = []
        self.step_rewards = []

    def _on_step(self) -> bool:
        obs = self.locals['obs'].copy()
        self.observations.append(obs)
        self.step_rewards.append((self.num_timesteps, self.locals['rewards'][0]))
        
        # Print observation shape and sample values
        if self.num_timesteps % 100 == 0:  # Print every 100 steps for example
            print(f"Step: {self.num_timesteps}, Observation shape: {obs.shape}")
            print(f"Sample observation: {obs[:5]}")  # Print first 5 values
        
        return True

    def _on_training_end(self) -> None:
        pass


    ARS (Augmented Randome Search) Model

In [ ]:
class HP():
    def __init__(self,
                 nb_steps=1000,
                 episode_length=2000,
                 learning_rate=0.02,
                 num_deltas=16,
                 num_best_deltas=16,
                 noise=0.03,
                 seed=1,
                 env_name='BipedalWalker4-v3',
                 record_every=300):

        self.nb_steps = nb_steps
        self.episode_length = episode_length
        self.learning_rate = learning_rate
        self.num_deltas = num_deltas
        self.num_best_deltas = num_best_deltas
        assert self.num_best_deltas <= self.num_deltas
        self.noise = noise
        self.seed = seed
        self.env_name = env_name
        self.record_every = record_every

class Normalizer():
    def __init__(self, nb_inputs):
        self.n = 0
        self.mean = np.zeros(nb_inputs)
        self.mean_diff = np.zeros(nb_inputs)
        self.var = np.zeros(nb_inputs)

    def observe(self, x):
        x = self.flatten_observation(x)
        self.n += 1.0
        last_mean = self.mean.copy()
        self.mean += (x - self.mean) / self.n
        self.mean_diff += (x - last_mean) * (x - self.mean)
        self.var = (self.mean_diff / self.n).clip(min=1e-2)

    def normalize(self, inputs):
        obs_mean = self.mean
        obs_std = np.sqrt(self.var)
        return (inputs - obs_mean) / obs_std

    @staticmethod
    def flatten_observation(observation):
        if isinstance(observation, np.ndarray):
            return observation.flatten()
        elif isinstance(observation, (list, tuple)):
            if len(observation) == 0:
                return np.array([])
            return np.concatenate([Normalizer.flatten_observation(x) for x in observation if x is not None])
        elif isinstance(observation, dict):
            if len(observation) == 0:
                return np.array([])
            return np.concatenate([Normalizer.flatten_observation(x) for x in observation.values() if x is not None])
        else:
            return np.array([observation])

class Policy():
    def __init__(self, input_size, output_size, hp):
        self.theta = np.zeros((output_size, input_size))
        self.hp = hp

    def evaluate(self, input, delta = None, direction = None):
        if direction is None:
            return self.theta.dot(input)
        elif direction == "+":
            return (self.theta + self.hp.noise * delta).dot(input)
        elif direction == "-":
            return (self.theta - self.hp.noise * delta).dot(input)

    def sample_deltas(self):
        return [np.random.randn(*self.theta.shape) for _ in range(self.hp.num_deltas)]

    def update(self, rollouts, sigma_rewards):
        step = np.zeros(self.theta.shape)
        for r_pos, r_neg, delta in rollouts:
            step += (r_pos - r_neg) * delta
        self.theta += self.hp.learning_rate / (self.hp.num_best_deltas * sigma_rewards) * step

class ARSTrainer():
    def __init__(self, hp=None, input_size=None, output_size=None, normalizer=None, policy=None, monitor_dir=None, callback=None):
        self.hp = hp or HP()
        np.random.seed(self.hp.seed)
        self.env = gym.make(self.hp.env_name, render_mode="rgb_array")
        self.hp.episode_length = self.env.spec.max_episode_steps or self.hp.episode_length
        self.input_size = input_size or self.env.observation_space.shape[0]
        self.output_size = output_size or self.env.action_space.shape[0]
        self.normalizer = normalizer or Normalizer(self.input_size)
        self.policy = policy or Policy(self.input_size, self.output_size, self.hp)
        self.monitor_dir = monitor_dir
        self.callback = callback
        self.record_video = False
        self.rewards = []

    def explore(self, direction=None, delta=None, step=0):
        state, info = self.env.reset()
        done = False
        num_plays = 0.0
        sum_rewards = 0.0
        video_recorder = None
        if self.record_video and self.monitor_dir:
            video_recorder_path = os.path.join(self.monitor_dir, f"step_{step}.mp4")
            print(f"Starting video recording: {video_recorder_path}")
            try:
                video_recorder = VideoRecorder(self.env, path=video_recorder_path)
            except Exception as e:
                print(f"Error initializing video recorder: {e}")

        while not done and num_plays < self.hp.episode_length:
            self.normalizer.observe(state)
            state = self.normalizer.normalize(self.normalizer.flatten_observation(state))
            action = self.policy.evaluate(state, delta, direction)
            state, reward, done, truncated, _ = self.env.step(action)
            reward = max(min(reward, 1), -1)
            sum_rewards += reward
            num_plays += 1
            if video_recorder:
                video_recorder.capture_frame()

        if video_recorder:
            try:
                video_recorder.close()
                print(f"Finished video recording: {video_recorder_path}")
                if os.path.isfile(video_recorder_path):
                    print(f"Video successfully saved: {video_recorder_path}")
                else:
                    print(f"Failed to save video: {video_recorder_path}")
            except Exception as e:
                print(f"Error closing video recorder: {e}")

        if self.callback:
            self.callback.locals = {
                'obs': state,
                'rewards': [sum_rewards]
            }
            self.callback._on_step()

        return sum_rewards

    def train(self):
        for step in range(self.hp.nb_steps):
            deltas = self.policy.sample_deltas()
            positive_rewards = [0] * self.hp.num_deltas
            negative_rewards = [0] * self.hp.num_deltas

            for k in range(self.hp.num_deltas):
                positive_rewards[k] = self.explore(direction="+", delta=deltas[k], step=step)
                negative_rewards[k] = self.explore(direction="-", delta=deltas[k], step=step)
                
            sigma_rewards = np.array(positive_rewards + negative_rewards).std()
            scores = {k:max(r_pos,r_neg) for k,(r_pos,r_neg) in enumerate(zip(positive_rewards, negative_rewards))}
            order = sorted(scores.keys(), key=lambda x:scores[x], reverse=True)[:self.hp.num_best_deltas]
            rollouts = [(positive_rewards[k], negative_rewards[k], deltas[k]) for k in order]
            self.policy.update(rollouts, sigma_rewards)

            if step % self.hp.record_every == 0:
                self.record_video = True
                print(f"Recording video at step: {step}")
            else:
                self.record_video = False

            reward_evaluation = self.explore(step=step)
            self.rewards.append(reward_evaluation)
            print('Step: ', step, 'Reward: ', reward_evaluation)

        if self.callback:
            self.callback._on_training_end()
# Create directories for storing videos
def mkdir(base, name):
    path = os.path.join(base, name)
    if not os.path.exists(path):
        os.makedirs(path)
    return path

    Record Video for Rendering

In [ ]:
# Create directories for videos
base_dir = '.'
videos_dir = mkdir(base_dir, 'videos_imposter4')
monitor_dir = mkdir(videos_dir, 'BipedalWalker4-v3')

In [ ]:
# Initialize the custom environment and record videos
env = BipedalWalker4()
env = RecordVideo(env, video_folder=monitor_dir, episode_trigger=lambda episode_id: True)

In [ ]:
# Initialize hyperparameters and trainer
hp = HP(env_name='BipedalWalker4-v3')
# Initialize the custom callback
callback = CustomCallback(env)
trainer = ARSTrainer(hp=hp, monitor_dir=monitor_dir, callback=callback)

# Train the model with the custom callback
trainer.train()
# Save the observations (example visualization)
obs_array = np.array(callback.observations)
obs_array = obs_array.reshape(obs_array.shape[0], -1)  # Reshape to 2D
obs_df = pd.DataFrame(obs_array)
step_rewards = pd.DataFrame(callback.step_rewards, columns=['step', 'reward'])
obs_df.to_csv('training_data_imposter4.csv', index=False)
step_rewards.to_csv('step_rewards_imposter4.csv', index=False)

    Evaluate Model

In [ ]:
# Evaluation function
def evaluate_model(env, policy, normalizer, num_episodes=10):
    total_rewards = []
    for episode in range(num_episodes):
        obs, _ = env.reset()
        done = False
        episode_reward = 0
        while not done:
            # Normalize the observation
            normalizer.observe(obs)
            obs = normalizer.normalize(normalizer.flatten_observation(obs))
            # Get the action from the policy
            action = policy.evaluate(obs)
            # Step the environment
            obs, reward, done, truncated, _ = env.step(action)
            episode_reward += reward
        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1}: Reward = {episode_reward}")
    mean_reward = np.mean(total_rewards)
    std_reward = np.std(total_rewards)
    print(f"Mean reward over {num_episodes} episodes: {mean_reward}")
    print(f"Standard deviation of reward over {num_episodes} episodes: {std_reward}")
    return mean_reward, std_reward

# Initialize the environment
env = gym.make('BipedalWalker4-v3', render_mode="rgb_array")

# Create an instance of HP, Normalizer, and Policy with the trained parameters
hp = HP()
normalizer = Normalizer(nb_inputs=env.observation_space.shape[0])
policy = Policy(input_size=env.observation_space.shape[0], output_size=env.action_space.shape[0], hp=hp)

# Assuming `trained_policy` and `trained_normalizer` are the objects obtained after training
trained_policy = policy  # Replace with your trained policy
trained_normalizer = normalizer  # Replace with your trained normalizer

# Evaluate the model
mean_reward, std_reward = evaluate_model(env, trained_policy, trained_normalizer, num_episodes=10)

# Close the environment
env.close()

In [ ]:
import pandas as pd

# Load the step rewards data
step_rewards = pd.read_csv('step_rewards_imposter4.csv')

# Calculate the rolling mean of rewards over the last 100 steps
rolling_mean_rewards = step_rewards['reward'].rolling(window=100).mean()

# Get the last rolling mean reward
last_rolling_mean = rolling_mean_rewards.iloc[-1]

# Check if the environment is solved
solved = last_rolling_mean >= 300

# Print the results
print(f"The last rolling mean reward (over 100 steps) is: {last_rolling_mean}")
print(f"Environment solved: {solved}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the step rewards data
step_rewards = pd.read_csv('step_rewards_imposter4.csv')

# Print the first few rows to check the data
print(step_rewards.head())

# Fix the step values assuming each row represents a single step
step_rewards['step'] = step_rewards.index

# Calculate the rolling mean of rewards over the last 100 steps
rolling_mean_rewards = step_rewards['reward'].rolling(window=100).mean()

# Get the rewards and steps
rewards = step_rewards['reward']
steps = step_rewards['step']

# Plot the step rewards and rolling mean rewards over time
plt.figure(figsize=(14, 7))

# Plot the rewards
plt.plot(steps, rewards, label='Step Reward', color='blue', alpha=0.3)

# Plot the rolling mean rewards
plt.plot(steps, rolling_mean_rewards, label='Rolling Mean Reward (100 steps)', color='red')

# Add the solved threshold line
plt.axhline(y=300, color='green', linestyle='--', label='Solved Threshold (300)')

# Add labels and title
plt.title('Step Rewards and Rolling Mean Rewards over Time')
plt.xlabel('Steps')
plt.ylabel('Score')
plt.legend()

# Show the plot
plt.show()

# Save the corrected step_rewards data if needed
step_rewards.to_csv('corrected_step_rewards4.csv', index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the rewards data
reward_df = pd.read_csv('step_rewards_imposter4.csv')

# Convert the reward column from string representation of lists to floats
# reward_df['reward'] = reward_df['reward'].apply(lambda x: float(x.strip('[]')))

# Calculate the rolling mean reward
reward_df['rolling_mean_reward'] = reward_df['reward'].rolling(window=100).mean()

# Mean reward and standard deviation from evaluation
# mean_reward = -616.47
# std_reward = 402.14

# Plot the results
plt.figure(figsize=(14, 7))
plt.plot(reward_df.index, reward_df['reward'], label='Step Reward', alpha=0.3, color='blue')
plt.plot(reward_df.index, reward_df['rolling_mean_reward'], label='Rolling Mean Reward (100 steps)', color='red')
plt.axhline(y=300, color='green', linestyle='--', label='Solved Threshold (300)')
plt.axhline(y=mean_reward, color='orange', linestyle='--', label=f'Mean Eval Reward ({mean_reward})')
plt.fill_between(reward_df.index, mean_reward - std_reward, mean_reward + std_reward, color='orange', alpha=0.2, label=f'Std Dev ({std_reward})')
plt.xlabel('Steps')
plt.ylabel('Score')
plt.title('Step Rewards and Rolling Mean Rewards over Time')
plt.legend()
plt.grid(True)

# Save the plot
plt.savefig('step_rewards_plot_with_mean_eval.png')
plt.show()


In [ ]:
# Check the contents of the videos directory
video_files = os.listdir(monitor_dir)
print("Generated video files:", video_files)

# List all video files
video_files = glob.glob(f"videos/{ENV_NAME}/step_*.mp4")

# Display download links for each video file
from IPython.display import display, FileLink

for file in video_files:
    display(FileLink(file))



    Entropy

In [ ]:
import pandas as pd
obs_df = pd.read_csv('training_data_imposter4.csv')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
entropies = []
from scipy.stats import entropy
# Iterate over the columns of your DataFrame
for feature in obs_df.columns:
    # Calculate the PDF of the Gaussian distribution for each column
    entropy = stats.entropy(np.histogram(obs_df[feature], bins=10)[0])
    print(f"Entropy for '{feature}': {entropy:.4f}")
    entropies.append(entropy)

In [ ]:
combined_single_entropy = sum(entropies)
print(f'Combined Single Entropy (Sum): {combined_single_entropy:.4f}')

In [ ]:
#Mean
mean = sum(entropies) / len(entropies)
print(mean)
#Standard Deviation
std_dev = np.std(entropies)
print(std_dev)

In [ ]:
alpha = 1
threshold_entropy = mean + alpha*std_dev
print(threshold_entropy)

In [ ]:
# Plot the line graph
plt.plot(obs_df.columns, entropies, marker='o')
plt.xlabel('Features')
plt.ylabel('Entropy')
plt.title('Entropy of all 9 Features')
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Bar Graph
import matplotlib.pyplot as plt

# Indices to highlight
highlight_indices = [14, 15]  # Replace with actual indices to highlight

# Set colors based on indices
colors = ['green' if i in highlight_indices else 'blue' for i in range(len(entropies))]


plt.bar(obs_df.columns, entropies, color=colors)
plt.xlabel('Features')
plt.ylabel('Entropy')
plt.title('Entropy of all 9 Features')
plt.xticks(rotation=45)
plt.show()

    KL Divergence

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from scipy.stats import entropy

# Assuming obs_df is your DataFrame with features
feature_columns = obs_df.columns

# Generate all combinations of features
feature_combinations = list(combinations(feature_columns, 2))

# Initialize lists to store feature pairs and their KL divergences
feature_pairs = []
kl_divergences = []

# Calculate KL divergences
for feature1, feature2 in feature_combinations:
    # Calculate histograms for the two features
    hist_1, _ = np.histogram(obs_df[feature1], bins=20, density=True)
    hist_2, _ = np.histogram(obs_df[feature2], bins=20, density=True)

    # Calculate probability distributions
    dist_1 = hist_1 / np.sum(hist_1)
    dist_2 = hist_2 / np.sum(hist_2)

    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10

    # Calculate KL divergence
    kl_divergence = np.sum(dist_1 * np.log((dist_1 + epsilon) / (dist_2 + epsilon)))

    # Store feature pair and KL divergence
    feature_pairs.append(f"{feature1} - {feature2}")
    kl_divergences.append(kl_divergence)

# Create a bar chart
plt.figure(figsize=(12, 6))
plt.bar(feature_pairs, kl_divergences, color='blue')
plt.xlabel('Feature Pairs')
plt.ylabel('KL Divergence')
plt.title('KL Divergence Between Feature Pairs')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
#Mean
mean = sum(kl_divergences) / len(kl_divergences)
print(mean)

In [ ]:
from itertools import combinations
from scipy.stats import entropy
import numpy as np

# Assuming obs_df is your DataFrame with features
feature_columns = obs_df.columns

# Generate all combinations of features
feature_combinations = list(combinations(feature_columns, 2))

for feature1, feature2 in feature_combinations:
    # Calculate histograms for the two features
    hist_1, _ = np.histogram(obs_df[feature1], bins=20, density=True)
    hist_2, _ = np.histogram(obs_df[feature2], bins=20, density=True)

    # Calculate probability distributions
    dist_1 = hist_1 / np.sum(hist_1)
    dist_2 = hist_2 / np.sum(hist_2)

    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10

    # Calculate KL divergence
    kl_divergence = np.sum(dist_1 * np.log((dist_1 + epsilon) / (dist_2 + epsilon)))

    print(f"KL Divergence between {feature1} and {feature2}: {kl_divergence}")


    Joint Entropy

In [ ]:
import itertools
joint_entropies = []
features = obs_df.columns
# Generate all possible combinations of 2 features
combinations = list(itertools.combinations(features, 2))
print(len(combinations))
print(combinations)

In [ ]:
joint_entropies = []
joint_entropies_list = []
for feature1, feature2 in combinations:
    # Calculate the joint histogram
    hist, x_edges, y_edges = np.histogram2d(obs_df[feature1], obs_df[feature2], bins=[10, 10])
    # Calculate the joint probability distribution
    p = hist / np.sum(hist)
    joint_entropy = stats.entropy(p.flatten(), base=2)
    print(f"Joint Entropy for '{feature1}' and '{feature2}': {joint_entropy:.4f}")
    joint_entropies.append(joint_entropy)
    joint_entropies_list.append((feature1, feature2, joint_entropy))

In [ ]:
combined_joint_entropy = sum(joint_entropies)
print(combined_joint_entropy)

In [ ]:
#Mean
mean = sum(joint_entropies) / len(joint_entropies)
print(mean)

In [ ]:
# Plot the line graph
feature_pairs = [f"{feature1} - {feature2}" for feature1, feature2, _ in joint_entropies_list]
plt.plot(feature_pairs, joint_entropies, marker='o')
plt.xlabel('Feature Combinations')
plt.ylabel('Joint Entropy')
plt.title('Joint Entropy without y-coord')
plt.xticks(rotation=45)
plt.show()